# Polynomial Regression


Polynomial regression is a type of regression analysis in which the relationship between the independent variable x and the dependent variable y is modeled as a polynomial of degree n. It is an extension of linear regression, which only models linear relationships between variables.

In polynomial regression, the relationship between x and y is represented by a polynomial equation, which can capture more complex relationships, including curvature and other nonlinear patterns. The degree of the polynomial equation determines the level of complexity of the relationship.

The general form of a polynomial regression equation is:

$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_n x^n$

x: independent variable

y: dependent variable

$\beta_0$: intercept

$\beta_i$: coefficients/parameters of the regression model, $i \in [1,n]$

n: degree of the polynomial

We will implement this regression model with the Scikit-Learn "LinearRegression" library.


1) Import the necessary libraries such as numpy, pandas, matplotlib.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plots_dir = Path("../../plots/etude3_Reg_poly")
plots_dir.mkdir(parents=True, exist_ok=True)


2. Import the file data_RP.csv into a DataFrame.


In [ ]:
df = pd.read_csv("data_RP.csv").drop(columns=["Unnamed: 0"], errors="ignore")
df.head()


3. Data analysis: info, describe.


In [ ]:
display(df.info())
display(df.describe())


4) Plot the scatter plot of the dataset y(x).


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["x"], df["y"], color="steelblue")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Polynomial Regression Dataset")
fig.tight_layout()
fig.savefig(plots_dir / "01_scatter_y_x.pdf", bbox_inches="tight")
plt.show()


Note: The relationship y(x) is nonlinear, so we will use the polynomial regression model to represent it. First, we need to build the train and test datasets.


5. Define x and y, then split into train and test (test_size = 0.3).

Then verify the split percentage.


In [ ]:
X = df[["x"]]
y = df["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print(f"Train percentage: {len(X_train) / len(X):.1%}")
print(f"Test percentage: {len(X_test) / len(X):.1%}")


6. On one graph, represent the scatter plots of the training set and the test set using distinct colors.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_train["x"], y_train, label="Train", color="steelblue")
ax.scatter(X_test["x"], y_test, label="Test", color="orange")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Train/Test Split")
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "02_train_test_split.pdf", bbox_inches="tight")
plt.show()


7. What degree should the polynomial have to represent y(x) well?

Create this polynomial with PolynomialFeatures from Scikit-Learn: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html


In [ ]:
degree = 2
polynomial_features = PolynomialFeatures(degree=degree, include_bias=False)
polynomial_features


8. Create the new "independent" variables from x_train and the polynomial from question 7 using fit_transform.

Create a linear regression model: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

Train the model.


In [ ]:
X_train_poly = polynomial_features.fit_transform(X_train)
X_test_poly = polynomial_features.transform(X_test)
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
poly_model


9. Make predictions for the train and test sets.

For each dataset, display the score and the RMSE error.


In [ ]:
train_pred = poly_model.predict(X_train_poly)
test_pred = poly_model.predict(X_test_poly)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Train R2: {poly_model.score(X_train_poly, y_train):.4f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test R2: {poly_model.score(X_test_poly, y_test):.4f}")
print(f"Test RMSE: {test_rmse:.4f}")


10. Display the regression model coefficients: intercept_ and coef_.


In [ ]:
print(f"Intercept: {poly_model.intercept_:.6f}")
print(f"Coefficients: {poly_model.coef_}")


12. Graphically visualize the polynomial of the polynomial regression model.

Put the scatter plot of the y(x) database on the same graph.


In [ ]:
x_grid = np.linspace(X["x"].min(), X["x"].max(), 300).reshape(-1, 1)
x_grid_df = pd.DataFrame(x_grid, columns=["x"])
y_grid = poly_model.predict(polynomial_features.transform(x_grid_df))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["x"], df["y"], label="Observed data", color="steelblue")
ax.plot(x_grid[:, 0], y_grid, label=f"Polynomial degree {degree}", color="crimson")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Polynomial Regression Fit")
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "03_polynomial_fit_degree_2.pdf", bbox_inches="tight")
plt.show()


13. Why can the "linear regression" model be used to perform polynomial regression?


Answer: LinearRegression is linear in the model coefficients, not necessarily in the original variable x. After PolynomialFeatures transforms x into x, x², ..., xⁿ, the model remains a linear combination of these transformed features, so LinearRegression can estimate the polynomial coefficients.


14. Compare performance (RMSE and R² on the train and test sets) with the following 3 polynomial degrees: n = 1, n = 2, and n = 15.

What do you notice?


Answer: Degree 1 underfits because it cannot capture curvature. Degree 2 captures the main nonlinear shape well. Degree 15 can fit the training data very closely but may overfit and generalize worse on the test set.


In [ ]:
degree_results = []
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["x"], df["y"], color="lightgray", label="Observed data")

for d in [1, 2, 15]:
    pf = PolynomialFeatures(degree=d, include_bias=False)
    X_train_d = pf.fit_transform(X_train)
    X_test_d = pf.transform(X_test)
    model_d = LinearRegression().fit(X_train_d, y_train)
    train_pred_d = model_d.predict(X_train_d)
    test_pred_d = model_d.predict(X_test_d)
    grid_pred_d = model_d.predict(pf.transform(x_grid_df))

    degree_results.append({
        "degree": d,
        "train_R2": r2_score(y_train, train_pred_d),
        "test_R2": r2_score(y_test, test_pred_d),
        "train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred_d)),
        "test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred_d)),
    })
    ax.plot(x_grid[:, 0], grid_pred_d, label=f"degree {d}")

degree_results_df = pd.DataFrame(degree_results)
display(degree_results_df)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Polynomial Degree Comparison")
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "04_degree_comparison.pdf", bbox_inches="tight")
plt.show()
